# Définition des données

## Récupération des données

In [ ]:
import pathlib

!rm -rf sample_data

if not pathlib.Path("landscape").is_dir():
  !wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape

!mkdir -p to-annotate

ids = [10034, 10159, 10244, 1060, 1072, 11, 11105, 1223, 12326, 12971]
for i, id in enumerate(ids):
  !cp landscape/seg_pred/no_supervision/{id}.jpg to-annotate/{i}.jpg

## Création d'instructions d'annotation

En étudiant les données présentes dans `landscape/seg_train`, créez des instructions d'annotation pour qu'en observant une image, un annotateur sache quelle classe attribuer parmi les 6 : `buildings` (bâtiments), `forest` (forêt), `glacier`, `mountain` (montagne), `sea` (mer) et `street` (rue).

La cellule suivante affiche quelques images de chaque classe, tirées au hasard.

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image

# Six images tirées au hasard dans chaque classe du jeu d'entraînement
random.seed(0)
classes = sorted(path.name for path in pathlib.Path("landscape/seg_train").iterdir())
fig, axes = plt.subplots(len(classes), 6, figsize=(12, 2.2 * len(classes)))
for row, name in zip(axes, classes):
  paths = random.sample(sorted(pathlib.Path("landscape/seg_train", name).glob("*.jpg")), 6)
  for ax, path in zip(row, paths):
    ax.imshow(Image.open(path))
    ax.axis("off")
  row[0].set_title(name, loc="left", fontweight="bold")
plt.tight_layout()

*Vos instructions*

## Annotation

Utilisez les instructions d'annotation d'une autre personne en les respectant le plus possible pour annoter les images dans le dossier `to-annotate`.

Remplissez la liste `labels` avec les classes que vous avez décidé d'attribuer à chacune des images.

Les 10 images à annoter, numérotées de 0 à 9 dans l'ordre où `labels` doit les lister :

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.flat, range(10)):
  ax.imshow(Image.open(f"to-annotate/{i}.jpg"))
  ax.set_title(f"Image {i}")
  ax.axis("off")
plt.tight_layout()

In [ ]:
num_classes = 6
class_to_index = dict(buildings=0, forest=1, glacier=2, mountain=3, sea=4, street=5)

labels = []  # À compléter

## Calcul de l'accord inter-annotateur

Mettez vos annotations en commun avec celles des autres participants, sur le support que le formateur indique (un tableur partagé ou un formulaire, par exemple), puis utilisez le coefficient kappa de Fleiss pour calculer l'accord inter-annotateur. C'est une extension du coefficient kappa de Cohen au cas où il y a plus de deux annotateurs. Il mesure l'accord entre annotateurs&nbsp;:

- Le score est compris entre -1 et 1
- 0 est un accord qui correspond au hasard
- En dessous de 0 les annotateurs sont moins d'accord qu'avec un tirage aléatoire des classes
- Inversement au dessus
- L'interprétation de ce score dépend du nombre de classes et est préférablement relative plutôt qu'absolue

Vous pourrez le calculer avec la méthode [`statsmodels.stats.inter_rater.fleiss_kappa`](https://www.statsmodels.org/stable/generated/statsmodels.stats.inter_rater.fleiss_kappa.html) ou en implémentant vous même les calculs (détaillés par exemple [sur Wikipédia](https://fr.wikipedia.org/wiki/Kappa_de_Fleiss)).

In [ ]:
#  Votre code ici

### Solution

In [ ]:
import pathlib

import numpy
import statsmodels.stats.inter_rater


def one_hot_encode(answers, num_classes=num_classes):
  return numpy.eye(num_classes)[answers]


all_labels = [
  # Une liste par annotateur, dans l'ordre des images (la vôtre est `labels`).
  # Exemple, à remplacer par les annotations du groupe :
  ["glacier", "buildings", "sea", "mountain", "sea", "glacier", "forest", "mountain", "glacier", "sea"],
  ["glacier", "street", "mountain", "glacier", "sea", "mountain", "street", "mountain", "glacier", "street"],
  ["sea", "buildings", "mountain", "glacier", "street", "mountain", "forest", "glacier", "glacier", "sea"],
]

n_files = len(list(pathlib.Path("to-annotate").glob("*.jpg")))

total = numpy.zeros((n_files, num_classes))
for answer in all_labels:
  total += one_hot_encode([class_to_index[name] for name in answer])

statsmodels.stats.inter_rater.fleiss_kappa(total)

## Impact sur le niveau de performance humain

Le niveau de performance humain (*Human Level Performance — HLP*), est très utilisé en Machine Learning. Quel est l'impact d'une mauvaise annotation sur les HLP ? Un algorithme d'apprentissage subit-il les mauvaises annotations de la même manière ?

*Votre réponse ici*

### Solution

Quelques éléments de réponse :

- Le HLP se mesure en comparant les annotations d'un humain à une référence (ou à celles des autres annotateurs). Avec un guide ambigu, des annotateurs compétents se contredisent : le HLP mesuré baisse, mais il reflète alors l'incohérence des étiquettes plus que la difficulté réelle de la tâche. Il estime donc mal l'erreur de Bayes.
- Un modèle peut alors sembler « dépasser le niveau humain » simplement parce qu'il reproduit mieux l'étiquette majoritaire. Ce n'est pas un vrai progrès : clarifier le guide fait remonter le HLP et révèle la marge qui reste.
- Un algorithme d'apprentissage ne subit pas toutes les erreurs de la même manière :
  - des erreurs aléatoires et peu nombreuses se compensent en partie sur beaucoup d'exemples, mais elles demandent plus de données et abaissent les performances atteignables ;
  - des erreurs systématiques (par exemple, toujours appeler `glacier` une montagne enneigée) sont apprises telles quelles : le modèle reproduit fidèlement la règle implicite des annotateurs ;
  - des étiquettes bruitées dans le jeu de test rendent l'évaluation elle-même peu fiable.
- Améliorer la cohérence de l'annotation (guide, double annotation des cas ambigus) est souvent plus rentable que de changer de modèle.